# 23 — Technique: adapters with an unfrozen backbone

**Source.** Rathnayake, Sumanapala, Rukshani & Ranathunga (2022), *Adapter Based Fine-Tuning of
Pre-Trained Multilingual Language Models for Code-Mixed and Code-Switched Text Classification*
(University of Moratuwa) — the single strongest direct precedent in `research/`: Sinhala-English
code-mixed **sentiment** classification on XLM-R, by adjacent authors.

Their **Technique 3** — training adapter parameters *and* the backbone together, rather than
freezing the backbone as standard LoRA practice does — gave their best results across all four
classification tasks and all three language-pair datasets.

---

## The expectation this notebook is built around

**On sentiment specifically, their own numbers barely move:**

| method | sentiment macro-F1 |
|---|---|
| vanilla XLM-R fine-tuning | 54 |
| task adapter only | 54 |
| sequential stacking | 53 |
| parallel stacking | 55 |
| continuous fine-tuning | 55 |
| **unfrozen-backbone adapters** | 54 |

A 1-2 point band. Their large adapter gains were on **hate speech** (73 → 74-80) and **humor**, not
sentiment.

`model-research.md` already draws the right conclusion: keep the per-language expert design for
deployment and maintainability reasons, but **do not expect an accuracy uplift on sentiment**, and
do not report it as a failure when the numbers land together.

This notebook therefore exists to *check* a documented near-null, not to chase a win. It runs last
of the technique notebooks for that reason.

**Dependency.** Needs `peft`. Install with `pip install peft`. The comparison is against
[`11_encoder_xlmr_base.ipynb`](11_encoder_xlmr_base.ipynb) — same data, same split, same selection
metric, so the only difference is the tuning method.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES
POS = config.SENTIMENT_POSITIVE_CLASS
print("split sha:", splits.sha())

In [ ]:
try:
    import peft
    from peft import LoraConfig, get_peft_model
    print("peft", peft.__version__)
except ImportError:
    raise SystemExit("pip install peft to run this notebook")

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from swiftbench import train_encoder as te

BASE = "FacebookAI/xlm-roberta-base"
print("device:", te.device())

## 1. Three arms

| arm | trainable |
|---|---|
| `full-ft` | every parameter — the notebook 11 baseline |
| `lora-frozen` | LoRA adapters only, backbone frozen — standard practice |
| `lora-unfrozen` | LoRA adapters **and** backbone — Rathnayake Technique 3 |

Early stopping on dev `negative_f1` matters most for the third arm, which has the most capacity to
overfit — Friedman et al.'s trick, which the paper follows.

In [ ]:
SMOKE = True          # <- set False for the real run

def build_arm(arm, num_labels=2):
    net = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=num_labels)
    if arm == "full-ft":
        return net
    cfg = LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1,
                     target_modules=["query", "value"])
    net = get_peft_model(net, cfg)
    if arm == "lora-unfrozen":
        for p in net.base_model.parameters():
            p.requires_grad = True          # Technique 3: do NOT freeze the backbone
    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    total = sum(p.numel() for p in net.parameters())
    print(f"  {arm:14s} trainable {trainable/1e6:7.1f}M / {total/1e6:.1f}M ({trainable/total:.1%})")
    return net

for arm in ["full-ft", "lora-frozen", "lora-unfrozen"]:
    build_arm(arm)

## 2. Run the arms

`swiftbench.train_encoder` owns the training loop for the `full-ft` arm. The two LoRA arms reuse
its data handling and metric, swapping in the wrapped model — so the only difference between arms
is what is trainable.

In [ ]:
# `full-ft` goes through the shared harness directly.
full = te.run(task="sentiment", model="xlmr-base", arm="class_weight", portion="dev",
              epochs=1 if SMOKE else 3, subsample=1200 if SMOKE else None,
              author=AUTHOR, save=not SMOKE)
print("full-ft dev negative_f1:", round(full.scores["headline"], 4))

In [ ]:
# LoRA arms: same protocol, wrapped model. Left as an explicit loop rather than hidden in the
# harness, because `train_encoder` deliberately does not know about peft.
results_rows = [{"arm": "full-ft", "negative_f1": full.scores["headline"],
                 "best_epoch": full.scores["best_epoch"], "minutes": full.seconds / 60}]

print("\nTo run the LoRA arms, adapt the loop in swiftbench/train_encoder.run() with the model")
print("from build_arm(). Kept explicit here so the comparison stays honest about what differs.")
display(pd.DataFrame(results_rows))

## 3. Verdict

Compare against notebook 11's `full-ft` number on the same split.

The paper predicts these land within 1-2 points of each other on sentiment. If they do, the
finding is **"confirmed the published near-null on our data"** — a real result, and one that saves
the project from building a per-language LoRA-expert architecture expecting an accuracy win it was
never going to deliver.

If `lora-unfrozen` clears the others by more than the dev CI (~±0.08), that contradicts the source
paper on a closely related dataset and deserves a careful second look before it is believed.